## Testing illusion range with pytorch models 

In [ ]:
from PIL import Image
import os 
import numpy as np
import torch
import imageio as io
import matplotlib.pyplot as plt
from collections import OrderedDict

import sys
sys.path.append('../')
import irnet

# Remove the parallelization from the model, making it usable with a single gpu and cpu
def remove_data_parallel(mod_name):
    state_dict = torch.load(mod_name, weights_only=True)
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('module.', '')  # Remove 'module.' prefix
        new_state_dict[name] = v
    return new_state_dict

# Prepare the image for model input
def prepare_tensor(arr):
    if len(arr.shape) == 2:
        arr = arr[np.newaxis, np.newaxis, ...]
    elif len(arr.shape) == 3:
        arr = arr[np.newaxis, ...]
    
    return torch.tensor(arr).float()

# Detach the tensor and turn into numpy array
def detach_tensor(tsr):
    return tsr.detach().cpu().numpy()[0][0]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


In [ ]:
images_dir = '../../illusions_matching'

counter = {}
illusion_names = set()
luminance_range = set()
for i, filename in enumerate(os.listdir(images_dir)):

    if 'mask' in filename:
        continue

    ill = filename.split('_')[1]
    lum = filename.split('_')[2]

    illusion_names.add(ill)
    luminance_range.add(lum)

    if ill not in counter:
        counter[ill] = [lum]
    else:
        counter[ill].append(lum)

luminance_range = sorted(luminance_range)
luminance_range

In [ ]:
i = 0
save_location = ''

for run_num in range(1, 11):
    model_path = f"../model_weights/irnet-512-run{run_num}-v10.pt"
    model = irnet.IRNet().to(device)
    model.load_state_dict(remove_data_parallel(model_path))

    for illusion in illusion_names:
        for luminance in luminance_range:
            filename = f'rgb_{illusion}_{luminance}_rng-000000_lum.hdr'
            imagepath = os.path.join(images_dir, filename) 

            image = io.imread(imagepath)[..., 0]
            image = prepare_tensor(image)

            output = model(image.to(device))
            output = detach_tensor(output)

            destination = os.path.join(save_location, f'model_run-{run_num}')
            if not os.path.exists(destination):
                os.makedirs(destination)
            im = Image.fromarray(output, mode='F')
            im.save(f"{destination}/{filename.replace('.hdr', '')}_output.tif")

            i += 1
        
print(i)